<a href="https://colab.research.google.com/github/sleep-is-best/Ai/blob/main/beta_0.1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q transformers[sentencepiece] protobuf

In [ ]:
!pip install -q transformers[sentencepiece] protobuf

### 🏺 نظام التدريب الشامل للأبجدية الميسينية (Linear B)

In [ ]:
import pandas as pd
import torch
from transformers import TrOCRProcessor, VisionEncoderDecoderModel
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from PIL import Image
import cv2
import os

# تحديث الأبجدية الميسينية الشاملة بناءً على طلب المستخدم
full_mycenaean_mapping = {
    '𐀀': 'a', '𐀁': 'e', '𐀂': 'i', '𐀃': 'o', '𐀄': 'u',
    '𐀅': 'da', '𐀆': 'de', '𐀇': 'di', '𐀈': 'do', '𐀉': 'du',
    '𐀊': 'ja', '𐀋': 'je', '𐀍': 'jo', '𐀎': 'ju',
    '𐀏': 'ka', '𐀐': 'ke', '𐀑': 'ki', '𐀒': 'ko', '𐀓': 'ku',
    '𐀔': 'ma', '𐀕': 'me', '𐀖': 'mi', '𐀗': 'mo', '𐀘': 'mu',
    '𐀙': 'na', '𐀚': 'ne', '𐀛': 'ni', '𐀜': 'no', '𐀝': 'nu',
    '𐀞': 'pa', '𐀟': 'pe', '𐀠': 'pi', '𐀡': 'po', '𐀢': 'pu',
    '𐀣': 'qa', '𐀤': 'qe', '𐀥': 'qi', '𐀦': 'qo',
    '𐀨': 'ra', '𐀩': 're', '𐀪': 'ri', '𐀫': 'ro', '𐀬': 'ru',
    '𐀭': 'sa', '𐀮': 'se', '𐀯': 'si', '𐀰': 'so', '𐀱': 'su',
    '𐀲': 'ta', '𐀳': 'te', '𐀴': 'ti', '𐀵': 'to', '𐀶': 'tu',
    '𐀷': 'wa', '𐀸': 'we', '𐀹': 'wi', '𐀺': 'wo',
    '𐀼': 'za', '𐀽': 'ze', '𐀿': 'zo',
    '𐁀': 'a2', '𐁁': 'a3', '𐁂': 'au', '𐁃': 'dwe', '𐁄': 'dwo',
    '𐁅': 'nwa', '𐁇': 'pte', '𐁆': 'pu2', '𐁈': 'ra2', '𐁉': 'ra3',
    '𐁊': 'ro2', '𐁋': 'ta2', '𐁌': 'twe', '𐁍': 'two'
}

# إضافة الرموز الفاصلة
full_mycenaean_mapping['𐄁'] = 'separator'

data = {
    'file_name': ['اثار للغة الميسينية.png'],
    'text': ['𐀗𐀛𐄁𐀀𐀸𐀆𐄁𐀳𐀀𐄁𐀟𐀩𐀷𐀆𐀃𐀍𐄁𐀀𐀑𐀩ဃa𐄁']
}
df_labels = pd.DataFrame(data)
display(df_labels)
print(f"Total Characters mapped: {len(full_mycenaean_mapping)}")

,file_name,text
0,اثار للغة الميسينية.png,𐀗𐀛𐄁𐀀𐀸𐀆𐄁𐀳𐀀𐄁𐀟𐀩𐀷𐀆𐀃𐀍𐄁𐀀𐀑𐀩ဃa𐄁


Total Characters mapped: 75


In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

class MycenaeanDataset(Dataset):
    def __init__(self, df, processor):
        self.df = df
        self.processor = processor

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        img_path = os.path.join('/content', self.df['file_name'][idx])
        image = cv2.imread(img_path)
        if image is None: raise FileNotFoundError(f"Image not found at {img_path}")

        # المعالجة المسبقة لتحسين التعرف
        gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
        _, thresh = cv2.threshold(gray, 150, 255, cv2.THRESH_BINARY_INV)

        pixel_values = self.processor(Image.fromarray(thresh).convert("RGB"), return_tensors="pt").pixel_values
        labels = self.processor.tokenizer(self.df['text'][idx], padding="max_length", max_length=128).input_ids
        labels = [label if label != self.processor.tokenizer.pad_token_id else -100 for label in labels]

        return {"pixel_values": pixel_values.squeeze(), "labels": torch.tensor(labels)}

In [5]:
import sentencepiece
import transformers
from transformers import TrOCRProcessor, VisionEncoderDecoderModel, RobertaTokenizer, ViTImageProcessor
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from PIL import Image
import torch
import cv2
import os
import pandas as pd
from tqdm.auto import tqdm # استيراد tqdm

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# الأبجدية الميسينية الأصلية (Linear B Unicode)
full_mycenaean_mapping = {
    '𐀀': 'a', '𐀁': 'e', '𐀂': 'i', '𐀃': 'o', '𐀄': 'u',
    '𐀅': 'da', '𐀆': 'de', '𐀇': 'di', '𐀈': 'do', '𐀉': 'du',
    '𐀊': 'ja', '𐀋': 'je', '𐀍': 'jo', '𐀎': 'ju',
    '𐀏': 'ka', '𐀐': 'ke', '𐀑': 'ki', '𐀒': 'ko', '𐀓': 'ku',
    '𐀔': 'ma', '𐀕': 'me', '𐀖': 'mi', '𐀗': 'mo', '𐀘': 'mu',
    '𐀙': 'na', '𐀚': 'ne', '𐀛': 'ni', '𐀜': 'no', '𐀝': 'nu',
    '𐀞': 'pa', '𐀟': 'pe', '𐀠': 'pi', '𐀡': 'po', '𐀢': 'pu',
    '𐀣': 'qa', '𐀤': 'qe', '𐀥': 'qi', '𐀦': 'qo',
    '𐀨': 'ra', '𐀩': 're', '𐀪': 'ri', '𐀫': 'ro', '𐀬': 'ru',
    '𐀭': 'sa', '𐀮': 'se', '𐀯': 'si', '𐀰': 'so', '𐀱': 'su',
    '𐀲': 'ta', '𐀳': 'te', '𐀴': 'ti', '𐀵': 'to', '𐀶': 'tu',
    '𐀷': 'wa', '𐀸': 'we', '𐀹': 'wi', '𐀺': 'wo',
    '𐀼': 'za', '𐀽': 'ze', '𐀿': 'zo',
    '𐁀': 'a2', '𐁁': 'a3', '𐁂': 'au', '𐁃': 'dwe', '𐁄': 'dwo',
    '𐁅': 'nwa', '𐁇': 'pte', '𐁆': 'pu2', '𐁈': 'ra2', '𐁉': 'ra3',
    '𐁊': 'ro2', '𐁋': 'ta2', '𐁌': 'twe', '𐁍': 'two',
    '𐄁': 'separator',
    '𐄸': '81',  # Linear B sign B081/TA (fraction/weight)
    '𐄹': '82',  # Linear B sign B082/D (fraction/weight)
    '𐁖': '87',  # Linear B sign B087/JU
    'I': 'I',    # Roman numeral I, or another symbol
    'II': 'II'   # Roman numeral II, or another symbol
}

data = {
    'file_name': [
        '1',
        '2',
        '3',
        '4',
        '5',
        '6'
    ],
    'text': [
        '𐀆𐀵𐀕𐀵𐀡',
        '𐀏𐀨𐀁𐀪𐀥𐀈𐀜𐀇𐀏𐀲𐀆𐀸𐀥',
        '𐀞𐀂𐁖𐀅𐀲𐀒𐀷𐀕𐀿𐀁𐀇𐀅𐀏𐀒𐀺𐀕𐀁𐀇',
        '𐀢𐀠𐀊𐀰𐀷𐀕𐀿𐀁𐀕𐀿𐀁',
        '𐀛𐀑𐀍𐀒𐀜𐀯𐀊𐄸𐄹',
        '𐀵𐀁𐀐𐀮𐀯𐀞I𐀳𐀺𐀂𐀥II𐀀𐀩'
    ]
}
df_labels = pd.DataFrame(data)

# Save df_labels to a CSV file as requested for model training data
df_labels.to_csv('mycenaean_labels.csv', index=False)
print("تم حفظ البيانات الجديدة في 'mycenaean_labels.csv'")

class MycenaeanDataset(Dataset):
    def __init__(self, df, processor):
        self.df = df
        self.processor = processor
    def __len__(self):
        return len(self.df)
    def __getitem__(self, idx):
        # Updated img_path to include the correct directory and file extension
        # Changed from .png to .jpg based on user's context.
        img_path = os.path.join('/content/sample_data/IMADES', self.df['file_name'][idx] + '.jpg')
        image = cv2.imread(img_path)
        if image is None: raise FileNotFoundError(f'Image not found at {img_path}')
        gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
        thresh = cv2.adaptiveThreshold(gray, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C, cv2.THRESH_BINARY_INV, 11, 2)
        pixel_values = self.processor(Image.fromarray(thresh).convert('RGB'), return_tensors='pt').pixel_values
        labels = self.processor.tokenizer(self.df['text'][idx], padding='max_length', max_length=128).input_ids
        labels = [label if label != self.processor.tokenizer.pad_token_id else -100 for label in labels]
        return {'pixel_values': pixel_values.squeeze(), 'labels': torch.tensor(labels)}

tokenizer = RobertaTokenizer.from_pretrained('microsoft/trocr-base-handwritten', use_fast=False)
image_processor = ViTImageProcessor.from_pretrained('microsoft/trocr-base-handwritten')
new_tokens = list(full_mycenaean_mapping.keys())
tokenizer.add_tokens(new_tokens)

processor = TrOCRProcessor(image_processor=image_processor, tokenizer=tokenizer)
model = VisionEncoderDecoderModel.from_pretrained('microsoft/trocr-base-handwritten').to(device)
model.decoder.resize_token_embeddings(len(tokenizer))
model.config.vocab_size = len(tokenizer)
model.config.decoder_start_token_id = tokenizer.cls_token_id
model.config.pad_token_id = tokenizer.pad_token_id

dataset = MycenaeanDataset(df_labels, processor)
dataloader = DataLoader(dataset, batch_size=1, shuffle=True)
optimizer = AdamW(model.parameters(), lr=1e-4)

print('Starting training with authentic Linear B Unicode symbols...')
model.train()
for epoch in range(50):
    total_loss = 0
    for batch in tqdm(dataloader, desc=f'Epoch {epoch+1}/50'): # إضافة شريط التقدم هنا
        optimizer.zero_grad()
        outputs = model(pixel_values=batch['pixel_values'].to(device), labels=batch['labels'].to(device))
        loss = outputs.loss
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    if (epoch + 1) % 10 == 0:
        print(f'Epoch {epoch+1}/50 - Loss: {total_loss:.4f}')
print('Training complete!')

تم حفظ البيانات الجديدة في 'mycenaean_labels.csv'


Loading weights:   0%|          | 0/478 [00:00<?, ?it/s]

[transformers] VisionEncoderDecoderModel LOAD REPORT from: microsoft/trocr-base-handwritten
Key                         | Status  | 
----------------------------+---------+-
encoder.pooler.dense.weight | MISSING | 
encoder.pooler.dense.bias   | MISSING | 

Notes:
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Starting training with authentic Linear B Unicode symbols...


Epoch 1/50:   0%|          | 0/6 [00:00<?, ?it/s]

...

Epoch 47/50:   0%|          | 0/6 [00:00<?, ?it/s]

Epoch 48/50:   0%|          | 0/6 [00:00<?, ?it/s]

Epoch 49/50:   0%|          | 0/6 [00:00<?, ?it/s]

Epoch 50/50:   0%|          | 0/6 [00:00<?, ?it/s]

Epoch 50/50 - Loss: 21.4782
Training complete!


In [ ]:
model.eval()
image_path = '/content/اثار للغة الميسينية.png'
image = cv2.imread(image_path)
gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
thresh = cv2.adaptiveThreshold(gray, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C, cv2.THRESH_BINARY_INV, 11, 2)
pixel_values = processor(Image.fromarray(thresh).convert('RGB'), return_tensors='pt').pixel_values.to(device)

with torch.no_grad():
    generated_ids = model.generate(
        pixel_values,
        max_new_tokens=50,
        num_beams=5,
        repetition_penalty=3.0,
        length_penalty=1.0,
        early_stopping=True
    )
    generated_text = processor.tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]

print(f'Detected text (raw): {generated_text}')

decoded_output = []
for char in generated_text:
    if char in full_mycenaean_mapping:
        decoded_output.append(full_mycenaean_mapping[char])
    else:
        decoded_output.append(f'[{char}]')

print(f'Phonetic Translation: {"-".join(decoded_output)}')

Detected text (raw): 𐄁𐀩𐀀𐀆𐀗𐀸𐀛𐀳𐀑𐀺𐀟𐀃𐀷𐀍𐄁𐄁𐄁𐄁𐄁𐄁𐄁𐄁𐄁𐄁𐄁𐄁𐄁𐄁𐄁𐄁𐄁𐄁𐄁𐄁𐄁𐄁𐄁𐄁𐄁𐄁𐄁𐄁𐄁𐄁𐄁𐄁𐄁𐄁𐄁
Phonetic Translation: separator-re-a-de-mo-we-ni-te-ki-wo-pe-o-wa-jo-separator-separator-separator-separator-separator-separator-separator-separator-separator-separator-separator-separator-separator-separator-separator-separator-separator-separator-separator-separator-separator-separator-separator-separator-separator-separator-separator-separator-separator-separator-separator-separator-separator-separator-separator


### استخراج النصوص من جميع الصور

الآن، بعد تدريب النموذج، يمكننا استخدامه لاستخراج النصوص من جميع الصور في مجموعة البيانات `df_labels`.

In [6]:
model.eval()

image_directory = '/content/sample_data/IMADES'

print("\n--- استخراج النصوص من جميع الصور ---")
for index, row in df_labels.iterrows():
    file_name = row['file_name']
    true_text = row['text']
    # تم التعديل للبحث عن امتداد .jpg بدلاً من .png
    image_path = os.path.join(image_directory, file_name + '.jpg')

    if not os.path.exists(image_path):
        print(f"تحذير: الصورة {image_path} غير موجودة. تخطي هذه الصورة.")
        continue

    image = cv2.imread(image_path)
    if image is None:
        print(f"تحذير: تعذر تحميل الصورة {image_path}. تخطي هذه الصورة.")
        continue

    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    thresh = cv2.adaptiveThreshold(gray, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C, cv2.THRESH_BINARY_INV, 11, 2)
    pixel_values = processor(Image.fromarray(thresh).convert('RGB'), return_tensors='pt').pixel_values.to(device)

    with torch.no_grad():
        generated_ids = model.generate(
            pixel_values,
            max_new_tokens=50,
            num_beams=5,
            repetition_penalty=3.0,
            length_penalty=1.0,
            early_stopping=True
        )
        generated_text = processor.tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]

    decoded_output = []
    for char in generated_text:
        if char in full_mycenaean_mapping:
            decoded_output.append(full_mycenaean_mapping[char])
        else:
            decoded_output.append(f'[{char}]')

    print(f"\nملف الصورة: {file_name}.jpg")
    print(f"النص الحقيقي: {true_text}")
    print(f"النص المكتشف (خام): {generated_text}")
    print(f"الترجمة الصوتية: {'-'.join(decoded_output)}")


--- استخراج النصوص من جميع الصور ---

ملف الصورة: 1.jpg
النص الحقيقي: 𐀆𐀵𐀕𐀵𐀡
النص المكتشف (خام): 𐀵𐀕𐀁𐀆𐀒𐀥𐀏𐀿𐀜𐀇𐀊𐀯𐀡𐀲𐀷𐀞𐀂𐀺𐀅𐄹𐄸𐀍𐀑𐀛𐀨𐀸𐀈𐀢𐀠𐀪𐀰𐀳𐀩𐀮𐀀𐀐𐁖III
الترجمة الصوتية: to-me-e-de-ko-qi-ka-zo-no-di-ja-si-po-ta-wa-pa-i-wo-da-82-81-jo-ki-ni-ra-we-do-pu-pi-ri-so-te-re-se-a-ke-87-I-I-I

ملف الصورة: 2.jpg
النص الحقيقي: 𐀏𐀨𐀁𐀪𐀥𐀈𐀜𐀇𐀏𐀲𐀆𐀸𐀥
النص المكتشف (خام): 𐀕𐀵𐀁𐀆𐀒𐀥𐀏𐀿𐀜𐀇𐀊𐀯𐀡𐀲𐀷𐀞𐀂𐀺𐀅𐄹𐄸𐀍𐀑𐀛𐀨𐀸𐀈𐀢𐀠𐀪𐀰𐀳𐀩𐀮𐀐𐀀𐁖III
الترجمة الصوتية: me-to-e-de-ko-qi-ka-zo-no-di-ja-si-po-ta-wa-pa-i-wo-da-82-81-jo-ki-ni-ra-we-do-pu-pi-ri-so-te-re-se-ke-a-87-I-I-I

ملف الصورة: 3.jpg
النص الحقيقي: 𐀞𐀂𐁖𐀅𐀲𐀒𐀷𐀕𐀿𐀁𐀇𐀅𐀏𐀒𐀺𐀕𐀁𐀇
النص المكتشف (خام): 𐀵𐀕𐀁𐀆𐀒𐀥𐀏𐀿𐀜𐀇𐀊𐀯𐀡𐀲𐀷𐀞𐀂𐀺𐀅𐄹𐄸𐀍𐀑𐀛𐀨𐀈𐀸𐀠𐀢𐀪𐀰𐀳𐀩𐀮𐀐𐀀𐁖III
الترجمة الصوتية: to-me-e-de-ko-qi-ka-zo-no-di-ja-si-po-ta-wa-pa-i-wo-da-82-81-jo-ki-ni-ra-do-we-pi-pu-ri-so-te-re-se-ke-a-87-I-I-I

ملف الصورة: 4.jpg
النص الحقيقي: 𐀢𐀠𐀊𐀰𐀷𐀕𐀿𐀁𐀕𐀿𐀁
النص المكتشف (خام): 𐀕𐀁𐀵𐀆𐀒𐀥𐀏𐀿𐀜𐀇𐀊𐀯𐀡𐀲𐀷𐀞𐀂𐀺𐀅𐄹𐄸𐀍𐀑𐀛𐀨𐀸𐀈𐀠𐀢𐀪𐀰𐀳𐀩𐀮𐀐𐀀𐁖III
الترجمة الصوتية: me-e-to-de-ko-qi-ka-zo-no-di-ja-si-po-ta-wa-pa-i-wo-da-82-81-jo-ki-ni-ra-we-do-pi-pu-ri-so-te-re-se-ke-a-87-I-I-I

ملف الصورة

### 💾 حفظ النموذج المدرب
يمكنك حفظ النموذج والمعالج (Processor) محلياً لاستخدامهما لاحقاً دون الحاجة لإعادة التدريب.

In [7]:
# حفظ النموذج والمعالج
model.save_pretrained('./mycenaean_trocr_model')
processor.save_pretrained('./mycenaean_trocr_model')
print("تم حفظ النموذج في المجلد: ./mycenaean_trocr_model")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

تم حفظ النموذج في المجلد: ./mycenaean_trocr_model


### 🔄 كيفية تحميل النموذج المحفوظ لاحقاً
يمكنك استخدام الكود التالي لتحميل النموذج الذي قمت بحفظه مسبقاً.

In [ ]:
from transformers import TrOCRProcessor, VisionEncoderDecoderModel

# تحميل النموذج والمعالج من المجلد المحفوظ
model_path = './mycenaean_trocr_model'
loaded_processor = TrOCRProcessor.from_pretrained(model_path)
loaded_model = VisionEncoderDecoderModel.from_pretrained(model_path).to(device)

print("تم تحميل النموذج بنجاح وهو جاهز للاستخدام!")